[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saranabhani/rug-analysing-data/blob/main/week4/linear_regression.ipynb)

Reference: https://programminghistorian.org/en/lessons/linear-regression#overview-of-linear-regression

# Importing required packages

In [ ]:
import pandas as pd
from datetime import date as dt
from sklearn.feature_extraction import DictVectorizer
from collections import Counter
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import seaborn as sns
import matplotlib.pyplot as plt

# Loading metadata file

In [ ]:
!git clone https://github.com/saranabhani/rug-analysing-data.git
%cd rug-analysing-data/week4

In [ ]:
metadata_file = "metadata_file.csv"
df = pd.read_csv(metadata_file)

In [ ]:
len(df)

In [ ]:
df.head()

In [ ]:
data_directory = "corpus/"

# List of dictionaries

## Creating a list of term-count dictionaries of the book reviews

In [ ]:
list_of_dictionaries = []
for row in df.iterrows():
    file_path = ''.join([data_directory, row[1]['filename']])
    temp_df = pd.read_csv(file_path).dropna().reset_index(drop=True).set_index('term')
    mydict = temp_df['count'].to_dict()
    list_of_dictionaries.append(mydict)

len(list_of_dictionaries)

In [ ]:
list_of_dictionaries[0]

## Select the top 10000 terms in the book reviews

In [ ]:
def top_words(number, list_of_dicts):
    totals = {}
    for d in list_of_dicts:
        for k,v, in d.items():
            try:
                totals[k] += v
            except:
                totals[k] = v
    # convert to counter object to use most common
    totals = Counter(totals)
    return [i[0] for i in totals.most_common(number)]

def cull_list_of_dicts(term_list, list_of_dicts):
    results = []
    for d in list_of_dicts:
        result = {}
        for term in term_list:
            try:
                result[term] = d[term]
            except:
                pass
        results.append(result)
    return results

top_term_list = top_words(10000, list_of_dictionaries)
new_list_of_dicts = cull_list_of_dicts(top_term_list, list_of_dictionaries)

In [ ]:
len(list_of_dictionaries[0]) - len(new_list_of_dicts[0])

## Convert to document-term matrix

In [ ]:
dictionary_vectorizer = DictVectorizer()
document_term = dictionary_vectorizer.fit_transform(new_list_of_dicts)

## TF-IDF transformation

In [ ]:
tfidf = TfidfTransformer()
tfidf_vectors = tfidf.fit_transform(document_term)

Select the best 5000 features using f_regression, f-regression is a univariate feature selection method that selects the best features based on the correlation between the feature and the target.

In [ ]:
tfidf_vectors_new = SelectKBest(f_regression, k=5000).fit_transform(tfidf_vectors, df['yearDecimal'])

## Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(tfidf_vectors_new, df['yearDecimal'], test_size=0.33, random_state=31)

## Linear regression model training

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

## Predictions and evaluation

In [ ]:
results = lr.predict(X_test)

In [ ]:
print(f"R2 score: {r2_score(list(y_test), list(results))}")

In [ ]:
def r_square_scratch(true, predicted):
    # substract each predicted value from each true
    residuals = [a - b for a, b in zip(true, predicted)]
    # calculate the sum of squared differences between predicted and true
    mss = sum([i**2 for i in residuals])
    # calculate the mean true value
    mean_true = (sum(true))/len(true)
    # calculate the sum of squared differences of each true value minus the mean true value
    tss = sum([(i-mean_true)**2 for i in true])
    # calculate final r2 value
    return 1-(mss/tss)

In [ ]:
results_df = pd.DataFrame()
results_df['predicted'] = list(results)
results_df['actual'] = list(y_test)
results_df['residual'] = results_df['predicted'] - results_df['actual']
results_df = results_df.sort_values(by='residual').reset_index(drop=True)
results_df.describe()

In [ ]:
sns.histplot(data=results_df['residual'])
plt.title("Histogram of Linear Regression Residuals")

In [ ]:
plt.xlim(1906, 1925)
plt.ylim(1895, 1935)
plt.scatter(results_df['actual'], results_df['predicted'], alpha=.35)
plt.plot([1895, 1935], [1895, 1935], color='black')
plt.title("Plot of Linear Regression Test Values, Predicted vs. Actual")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.subplots_adjust(top=0.85)
plt.show()

## Intercept and Coefficients

In [ ]:
print(f"Intercept: {lr.intercept_}")

In [ ]:
features = SelectKBest(f_regression, k=5000).fit(tfidf_vectors, df['yearDecimal'])

selected = features.get_support()

features_df = pd.DataFrame()
features_df['term'] = dictionary_vectorizer.feature_names_
features_df['selected'] = selected
features_df = features_df.loc[features_df['selected'] == True]
features_df['coef'] = lr.coef_

coefficients = features_df.sort_values(by='coef', ascending=False).reset_index(drop=True)

In [ ]:
coefficients

In [ ]:
selected

In [ ]:
coefficients.iloc[0:50]

In [ ]:
coefficients.iloc[-25:]